# Ingesta de reparto de películas

Lee `movie_cast.json` (JSON multilínea) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/movie_cast` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee `movie_cast.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
movie_cast_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_cast.json")
    )
display(movie_cast_df)

In [0]:
movie_cast_df.columns

In [0]:
movie_cast_schema = StructType([
    StructField("castOrder", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True)
])
movie_cast_df = (
    spark.read
        .schema(movie_cast_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_cast.json")
)

### Transformación de columnas

Conserva `movie_id`, `person_id` y `character_name` (descarta `castOrder` y `genderId`) y agrega `ingestion_date` y `env`.

In [0]:
movie_cast_df = (
    add_ingestion_date(movie_cast_df)
        .select(
            col("movieId").alias("movie_id"),
            col("personId").alias("person_id"),
            col("characterName").alias("character_name"),
            lit(v_env).alias("env")
        )
)
display(movie_cast_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/movie_cast` en modo `overwrite`.

In [0]:
(
    movie_cast_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/movie_cast")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")